In [82]:
import pandas as pd
import numpy as np

In [83]:
# 주문 상세 데이터 로딩
oi = pd.read_csv("../data/order_items.csv")
oi.info()
# unit_price Na 해결 필요

<class 'pandas.DataFrame'>
RangeIndex: 500000 entries, 0 to 499999
Data columns (total 6 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   order_item_id  500000 non-null  int64  
 1   order_id       500000 non-null  int64  
 2   product_id     500000 non-null  int64  
 3   quantity       500000 non-null  int64  
 4   unit_price     484887 non-null  float64
 5   discount       500000 non-null  float64
dtypes: float64(2), int64(4)
memory usage: 22.9 MB


In [84]:
oi.describe()[["quantity", "unit_price","discount"]]
# quantity - 해결 필요

,quantity,unit_price,discount
count,500000.000000,484887.000000,500000.000000
mean,2.997008,50925.557707,0.249688
std,1.418193,48908.412198,0.144496
min,-2.000000,1000.000000,0.000000
25%,2.000000,15100.000000,0.120000
50%,3.000000,27000.000000,0.250000
75%,4.000000,73700.000000,0.370000
max,5.000000,291000.000000,0.500000


In [85]:
# 데이터 정제 : unit_price NaN 처리, quantity 음수 처리
oi = oi.dropna(subset=["unit_price"])   # unit_price 컬럼 기준으로 NaN 들어있는 행 삭제
oi = oi[oi["quantity"]>0]

In [86]:
oi.describe()

,order_item_id,order_id,product_id,quantity,unit_price,discount
count,484400.000000,484400.000000,484400.000000,484400.000000,484400.000000,484400.000000
mean,249917.885072,99958.400541,220.309806,3.001201,50926.236168,0.249776
std,144299.824554,57753.831877,150.594182,1.412894,48909.686132,0.144482
min,1.000000,1.000000,1.000000,1.000000,1000.000000,0.000000
25%,124938.750000,49940.750000,92.000000,2.000000,15100.000000,0.120000
50%,249887.500000,99925.000000,216.000000,3.000000,27000.000000,0.250000
75%,374917.250000,149944.000000,348.000000,4.000000,73700.000000,0.370000
max,499880.000000,200000.000000,497.000000,5.000000,291000.000000,0.500000


In [87]:
# 주문 금액(단가 * 수량 * (1-할인율)) 컬럼 생성
# oi2 = oi.assign(      # 원본 보호
    # amount=단가 * 수량 * (1-할인율)
# )

# 벡터 연산(원소별 연산) 사용해서 처리
# 메모리 절약(기존 oi에 추가하므로)
oi["amount"] = oi["unit_price"] * oi["quantity"] * (1-oi["discount"])

In [88]:
oi.head()

,order_item_id,order_id,product_id,quantity,unit_price,discount,amount
0,59256,114625,3,1,20300.0,0.05,19285.0
1,230587,66337,80,2,90000.0,0.45,99000.0
2,279813,163343,3,3,20300.0,0.23,46893.0
3,88487,180332,79,1,8600.0,0.38,5332.0
4,39240,174179,232,1,42600.0,0.31,29394.0


In [89]:
# 주문금액이 100000 이상(조건)이면 고액(True), 아니면(False) "일반" 등급을 저장하는 컬럼 생성
# 신규 컬럼 : grade
# np.where(조건식 자리, True일 때 값, False일 때 값). 이진 분류
oi["grade"] = np.where(oi["amount"]>100000, '고액','일반')

In [90]:
oi.grade.value_counts()

grade
일반    320448
고액    163952
Name: count, dtype: int64

In [91]:
# 다중 분류 : np.select(조건들, 값들, 기본값.). 조건 3개면 선택지도 3개
# 주문 금액 >= 200000 : 최고액, 주문금액 >= 50000: 고액, 나머지: 일반   (조건 2개, default: 1개.)
conditionlist=[oi["amount"]>=200000, oi["amount"]>=50000]
choicelist = ["최고액", "고액"]
oi["tier"] = np.select(conditionlist, choicelist, default="일반")
oi["tier"].value_counts()

tier
일반     214435
고액     182057
최고액     87908
Name: count, dtype: int64

In [92]:
sum(oi["tier"].value_counts())

484400

In [93]:
# apply: DataFrame에 적용할 수 있는 메소드.
# np.where 조건이 많아서 복잡해지면 코드 지저분해 짐.
# 금액과 할인율을 함께 보는 사용자 정의 라벨
# 데이터 생성
data = {
    "name" : ["홍길동", "고길동", "마이꼴"],
    "kor":[100, 90, 90],
    "eng":[50, 90, 100]
}
data

{'name': ['홍길동', '고길동', '마이꼴'], 'kor': [100, 90, 90], 'eng': [50, 90, 100]}

In [94]:
students = pd.DataFrame(data)
students

,name,kor,eng
0,홍길동,100,50
1,고길동,90,90
2,마이꼴,90,100


In [95]:
students.set_index("name",inplace=True) # inplace=True 해야 원본에 저장됨.

In [96]:
students

,kor,eng
name,,
홍길동,100,50
고길동,90,90
마이꼴,90,100


In [97]:
students.loc["홍길동"]

kor    100
eng     50
Name: 홍길동, dtype: int64

In [102]:
students.sum(axis=0), students.sum(axis=1)

(kor    280
 eng    240
 dtype: int64,
 name
 홍길동    150
 고길동    180
 마이꼴    190
 dtype: int64)

In [ ]:
# for 사용. 3번 반복
# students["tot"] = students.apply(lambda row:row["kor"]+row["eng"], axis=1)

# apply의 sum() 함수 사용
# students["tot"] = students[["kor", "eng"]].apply(sum,axis=1)

# 벡터 연산(원소별 연산) : 병렬처리
students["tot"] = students["kor"] + students["eng"] # 벡터 연산(병렬처리가 됨.)
# 단순 더하기 같은 문제는 apply보단 벡터 연산이 더 빠름.
# 대용량 처리에 apply 사용 X.


# sum() 사용
# students["tot"] = students[["kor","eng"]].sum(axis=1)
students

# 원소별 연산 > sum() > apply()

,kor,eng,tot
name,,,
홍길동,100,50,150
고길동,90,90,180
마이꼴,90,100,190


In [104]:
# 금액과 할인율을 함께 보는 사용자 정의 라벨
# apply(function, axis) 사용해서 주문 금액 100000이상이고 할인율이 0.3 이상일 때 '고액-대폭할인', 아니면 '기타'
def label(row):
    if row["amount"] >= 100000 and row["discount"] >= 0.3:
        return "고액-대폭할인"
    return "기타"

In [ ]:
oi.apply(label, axis=1).value_counts()      # discount, amount 필요 => axis=1
                                            # axis=1이니 한 행씩 label 함수에 들어감.

기타         427701
고액-대폭할인     56699
Name: count, dtype: int64

In [111]:
condition1 = oi["amount"] >= 100000
condition2 = oi["discount"] >= 0.3
np.where(condition1&condition2,"고액-대폭할인","기타")

array(['기타', '기타', '기타', ..., '기타', '기타', '기타'],
      shape=(484400,), dtype='<U7')